# Anime Recommender · Notebook 1: 資料下載與 EDA

**目標**:把原始 Kaggle 資料下載、清理、切分,然後存成下兩本 notebook 直接用的格式。

執行流程:
1. 掛載 Google Drive (artifacts 會存到 `MyDrive/anime-recsys/artifacts/`)
2. 上傳 Kaggle API token (一次性,5 分鐘設定)
3. 下載 CooperUnion 的 Anime Recommendations Database
4. EDA - 評分分佈、類型分佈、熱門作品
5. 清理 + 過濾稀疏的 user / item
6. 建立 id 對照表
7. Per-user 80/20 train/test 切分
8. 輸出 `anime_meta.parquet`、`train.parquet`、`test.parquet`、`mappings.pkl`、`user_history.pkl`、`demo_users.json`

> 在 Colab 上點 **Runtime → Run all** 一次跑完即可。預估 5–10 分鐘。

## Step 1 — 掛載 Google Drive 並建立 artifacts 資料夾

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ARTIFACTS_DIR = '/content/drive/MyDrive/anime-recsys/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f'Artifacts will be saved to: {ARTIFACTS_DIR}')

## Step 2 — 設定 Kaggle API

到 https://www.kaggle.com/settings/account → 找到 **API** 區塊 → 點 **Create New Token**,
會下載一個 `kaggle.json`。執行下面這個 cell 時上傳這個檔案即可。

(之後其他 notebook 不需要再上傳)

In [ ]:
import os, shutil
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('請上傳你的 kaggle.json:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API ready ✅')

## Step 3 — 下載資料集 (~110 MB,解壓後 ~250 MB)

In [ ]:
!pip install -q kaggle
!mkdir -p /content/data
!cd /content/data && kaggle datasets download -d CooperUnion/anime-recommendations-database --unzip --force
!ls -lh /content/data

## Step 4 — 載入資料

In [ ]:
import pandas as pd
import numpy as np

anime = pd.read_csv('/content/data/anime.csv')
rating = pd.read_csv('/content/data/rating.csv')

print(f'anime: {anime.shape}, columns: {list(anime.columns)}')
print(f'rating: {rating.shape}, columns: {list(rating.columns)}')
anime.head()

In [ ]:
rating.head()

## Step 5 — EDA

### 5.1 評分分佈\n\nrating = -1 代表使用者看過但沒給分;為了訓練「顯式回饋 (explicit feedback)」模型,這些之後會被過濾掉。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
rating['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#1F4E79')
ax.set_title('Rating distribution (-1 = watched but not rated)')
ax.set_xlabel('rating'); ax.set_ylabel('count')
plt.tight_layout(); plt.show()

### 5.2 動漫類型 (Genre) 分佈

In [ ]:
from collections import Counter

# genre 欄位是 'Action, Adventure, Drama' 這樣的字串,先 split
genre_counter = Counter()
for g in anime['genre'].dropna():
    for token in g.split(','):
        genre_counter[token.strip()] += 1

top_genres = pd.Series(genre_counter).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 5))
top_genres.plot(kind='bar', ax=ax, color='#2E75B6')
ax.set_title('Top 20 Genres')
ax.set_ylabel('# anime')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()
print(f'總共 {len(genre_counter)} 種 genre')

### 5.3 作品形式 (Type) 分佈

In [ ]:
anime['type'].value_counts().plot(kind='bar', figsize=(7,3), color='#1F4E79', title='Anime type')
plt.tight_layout(); plt.show()

### 5.4 全站評分前 20 高的作品

In [ ]:
anime.dropna(subset=['rating']).sort_values('rating', ascending=False).head(20)[['name','genre','type','rating','members']]

### 5.5 每位 user 評了幾部?

In [ ]:
user_counts = rating[rating['rating'] > 0].groupby('user_id').size()
print(user_counts.describe())
user_counts.clip(upper=200).hist(bins=50, figsize=(8,3))
plt.title('Distribution of #ratings per user (clipped at 200)')
plt.xlabel('# ratings'); plt.ylabel('# users')
plt.tight_layout(); plt.show()

## Step 6 — 清理 + 過濾

- 移除 `rating = -1` (沒有顯式分數)
- 只保留評過 >= 5 部的 user
- 只保留被 >= 5 個 user 評過的 anime
- 避免太稀疏導致 collaborative filtering 訓練困難

In [ ]:
df = rating[rating['rating'] > 0].copy()

for _ in range(2):  # 兩輪過濾讓 user/item 都穩定
    user_counts = df.groupby('user_id').size()
    df = df[df['user_id'].isin(user_counts[user_counts >= 5].index)]
    item_counts = df.groupby('anime_id').size()
    df = df[df['anime_id'].isin(item_counts[item_counts >= 5].index)]

print(f'過濾後:{len(df):,} 筆評分 / {df["user_id"].nunique():,} 位 user / {df["anime_id"].nunique():,} 部 anime')

## Step 7 — 建立 id 對照表

In [ ]:
user_ids = sorted(df['user_id'].unique())
anime_ids = sorted(df['anime_id'].unique())

user_id_to_idx = {u: i for i, u in enumerate(user_ids)}
idx_to_user_id = {i: u for u, i in user_id_to_idx.items()}
anime_id_to_idx = {a: i for i, a in enumerate(anime_ids)}
idx_to_anime_id = {i: a for a, i in anime_id_to_idx.items()}

print(f'#users = {len(user_ids):,}, #anime = {len(anime_ids):,}')

## Step 8 — Per-user 80/20 切分

對每個 user 隨機抽 20% 評分當 test;這樣每個 user 在 train / test 都有資料,
評估時才能拿 test 當 ground truth。

In [ ]:
from sklearn.model_selection import train_test_split

rng = np.random.RandomState(42)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_parts, test_parts = [], []
for uid, group in df.groupby('user_id', sort=False):
    if len(group) < 5:
        train_parts.append(group)
        continue
    tr, te = train_test_split(group, test_size=0.2, random_state=42)
    train_parts.append(tr); test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)
print(f'train: {len(train_df):,}  test: {len(test_df):,}')

## Step 9 — 建立 user_history (基於 train,不可使用 test 防 data leakage)

In [ ]:
user_history = (
    train_df.groupby('user_id')['anime_id'].apply(set).to_dict()
)
print(f'user_history 涵蓋 {len(user_history):,} 位 user')

## Step 10 — 挑 10 位 Demo User (評分多但不要極端)

In [ ]:
ratings_per_user = train_df.groupby('user_id').size().sort_values(ascending=False)
# 取「評分數 30‒200」之間的 user,排前面但避免極端 power user
candidates = ratings_per_user[(ratings_per_user >= 30) & (ratings_per_user <= 200)]
demo_users = candidates.head(10).index.tolist()
print('Demo users:', demo_users)

## Step 11 — 儲存所有 artifacts 到 Google Drive

In [ ]:
import pickle, json

# anime metadata (整份),只保留我們會用到的欄位
anime_meta = anime[anime['anime_id'].isin(anime_ids)].copy()
anime_meta = anime_meta[['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']]
anime_meta.to_parquet(f'{ARTIFACTS_DIR}/anime_meta.parquet', index=False)

# 訓練 / 測試 split
train_df.to_parquet(f'{ARTIFACTS_DIR}/train.parquet', index=False)
test_df.to_parquet(f'{ARTIFACTS_DIR}/test.parquet', index=False)

# id 對照表
mappings = {
    'user_id_to_idx': user_id_to_idx,
    'idx_to_user_id': idx_to_user_id,
    'anime_id_to_idx': anime_id_to_idx,
    'idx_to_anime_id': idx_to_anime_id,
}
with open(f'{ARTIFACTS_DIR}/mappings.pkl', 'wb') as f:
    pickle.dump(mappings, f)

# user_history (給本地端「篩掉已看過」用)
with open(f'{ARTIFACTS_DIR}/user_history.pkl', 'wb') as f:
    pickle.dump(user_history, f)

# demo user 清單
with open(f'{ARTIFACTS_DIR}/demo_users.json', 'w') as f:
    json.dump([int(u) for u in demo_users], f)

print('✅ Saved to:', ARTIFACTS_DIR)
!ls -lh {ARTIFACTS_DIR}

## ✅ 完成

產出:
- `anime_meta.parquet`、`train.parquet`、`test.parquet`
- `mappings.pkl`、`user_history.pkl`、`demo_users.json`

下一步:打開 **02_train_models.ipynb** 訓練三個模型。